In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, unix_timestamp, to_timestamp, expr, lag, when, split, row_number,hour, dayofweek,stddev,monotonically_increasing_id,
    sum as Fsum, floor, concat_ws, count as Fcount, min as Fmin, max as Fmax, avg as Favg, percentile_approx, coalesce, lit
)
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType
import pandas as pd
from pyspark.sql.functions import pandas_udf, PandasUDFType
from pyspark.sql.functions import hour, dayofweek
from pyspark.ml.feature import StringIndexer
import time

start_time = time.time()
# 初始化 Spark 会话
spark = SparkSession.builder \
    .appName("wucaishendata") \
    .config("spark.driver.memory", "200g") \
    .config("spark.executor.memory", "32g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()

# 读取数据
df = spark.read.csv("wucaishen202503.csv", header=True, inferSchema=True)
df = df.filter((col("flag") != -8.0) & (col("productid") != "B26"))

#原始数据需要的列
columns_to_keep = [
    'productid', 'loginname', 'billno', 'billtime',
    'account', 'cus_account', 'currency', 'slottype',
    'basepoint', 'result', 'cur_ip'
]
df = df.select(*columns_to_keep)

# 时间转换
df = df.withColumn("billtime_utc", to_timestamp((col("billtime") / 1e9).cast("long")))
df = df.withColumn("billtime", expr("from_utc_timestamp(billtime_utc, 'America/New_York')"))

# 排序窗口
# 添加唯一标识行 ID
df = df.withColumn("row_id", monotonically_increasing_id())

# payout + current_point
df = df.withColumn("payout", col("cus_account") + col("account"))
df = df.withColumn("current_point", col("basepoint") + col("cus_account"))
# 对 currency 做 label encoding
indexer = StringIndexer(inputCol="currency", outputCol="currency_label")
currency_model = indexer.fit(df)
df = currency_model.transform(df)
# 上一笔记录
w = Window.partitionBy("loginname").orderBy("billtime")
df = df.withColumn("prev_time", lag("billtime").over(w))

df = df.withColumn("prev_account", when(lag("account").over(w).isNotNull(), lag("account").over(w)).otherwise(0))
df = df.withColumn("prev_profit", when(lag("cus_account").over(w).isNotNull(), lag("cus_account").over(w)).otherwise(0))
df = df.withColumn("prev_payout", when((col("prev_account") + col("prev_profit")).isNotNull(), col("prev_account") + col("prev_profit")).otherwise(0))
df = df.withColumn("last_current_point", when(lag("basepoint").over(w).isNotNull(), lag("basepoint").over(w) + lag("cus_account").over(w)).otherwise(0))
df = df.withColumn("is_payout_gt0", when(col("payout") > 0, 1).otherwise(0))
df = df.withColumn("is_profit_gt0", when(col("cus_account") > 0, 1).otherwise(0))
# 衍生字段：delta_t
df = df.withColumn("delta_t", when(col("prev_time").isNull(), 0.0).otherwise((unix_timestamp("billtime") - unix_timestamp("prev_time")).cast("double")))

#delta_bet
df = df.withColumn(
    "delta_bet",
    when(col("prev_account").isNotNull(), col("account") - col("prev_account")).otherwise(0)
)

#delta_profit
df = df.withColumn(
    "delta_profit",
    when(col("prev_profit").isNotNull(), col("cus_account") - col("prev_profit")).otherwise(0)
)
# balance_change = 当前 basepoint - 上一笔 current_point
df = df.withColumn(
    "balance_change",
    when(col("last_current_point").isNotNull(), col("basepoint") - col("last_current_point")).otherwise(0)
)

# 识别充值与提现
df = df.withColumn("deposit", when(col("balance_change") > 0, col("balance_change")).otherwise(0))
df = df.withColumn("withdrawal", when(col("balance_change") < 0, -col("balance_change")).otherwise(0))

df = df.withColumn("rtp", when(col("account") != 0, col("payout") / col("account")).otherwise(None))
# 添加时段分类列
df = df.withColumn("hour_of_day", hour("billtime"))
df = df.withColumn("is_morning", when((col("hour_of_day") >= 6) & (col("hour_of_day") < 12), 1).otherwise(0))
df = df.withColumn("is_afternoon", when((col("hour_of_day") >= 12) & (col("hour_of_day") < 18), 1).otherwise(0))
df = df.withColumn("is_night", when((col("hour_of_day") >= 18) & (col("hour_of_day") <= 23), 1).otherwise(0))
df = df.withColumn("is_midnight", when((col("hour_of_day") >= 0) & (col("hour_of_day") < 6), 1).otherwise(0))

# 节假日（周末）
df = df.withColumn("is_weekend", when(dayofweek("billtime").isin([1, 7]), 1).otherwise(0))  # 1=Sunday, 7=Saturday
# 拆分 result 字段为 result_pos1 ~ result_pos15
df = df.withColumn("result_clean", expr("trim(BOTH ';' FROM result)"))
split_cols = split(col("result_clean"), ",")
for i in range(15):
    df = df.withColumn(f"result_pos{i+1}", split_cols.getItem(i).cast(IntegerType()))

# ========= 连续投注（streak） =========
@pandas_udf("row_id long, streak int", PandasUDFType.GROUPED_MAP)
def compute_streak_udf(pdf):
    pdf = pdf.sort_values("billtime")
    streaks = []
    streak = 0
    for dt in pdf["delta_t"]:
        if pd.isna(dt) or dt > 200:
            streak = 0
        else:
            streak += 1
        streaks.append(streak)
    pdf["streak"] = streaks
    return pdf[["row_id", "streak"]]

streak_df = df.select("row_id", "loginname", "billtime", "delta_t") \
              .groupby("loginname").apply(compute_streak_udf)
df = df.join(streak_df, on=["row_id"], how="left")

# ========= 连续赢钱/输钱 streak =========
@pandas_udf("row_id long, win_streak int, lose_streak int", PandasUDFType.GROUPED_MAP)
def compute_win_lose_streak(pdf):
    pdf = pdf.sort_values("billtime")
    win_streaks = []
    lose_streaks = []
    win_streak = 0
    lose_streak = 0
    for profit in pdf["cus_account"]:
        if profit > 0:
            win_streak += 1
            lose_streak = 0
        elif profit < 0:
            lose_streak += 1
            win_streak = 0
        else:
            win_streak = 0
            lose_streak = 0
        win_streaks.append(win_streak)
        lose_streaks.append(lose_streak)
    pdf["win_streak"] = win_streaks
    pdf["lose_streak"] = lose_streaks
    return pdf[["row_id", "win_streak", "lose_streak"]]

streak2_df = df.select("row_id", "loginname", "billtime", "cus_account") \
               .groupby("loginname").apply(compute_win_lose_streak)
df = df.join(streak2_df, on=["row_id"], how="left")


# ========== 基于 delta_t > 7 天 切断分组 ==========
df = df.withColumn("is_split", when(col("delta_t") > 604800, 1).otherwise(0))
split_window = Window.partitionBy("loginname").orderBy("billtime")
df = df.withColumn("group_index", Fsum("is_split").over(split_window))

# ========== 每个断组再按 40 条一分 ==========
block_window = Window.partitionBy("loginname", "group_index").orderBy("billtime")
df = df.withColumn("row_number_in_block", row_number().over(block_window))
df = df.withColumn("sub_index", floor((col("row_number_in_block") - 1) / 40))

# 最终 group_id
df = df.withColumn("group_id", concat_ws("_", col("loginname"), col("group_index"), col("sub_index")))

# ========== 分组统计 ==========
agg_exprs = [
    Fcount("*").alias("group_num"),
    
    Fmin("account").alias("bet_min"),
    Fmax("account").alias("bet_max"),
    Favg("account").alias("bet_mean"),
    percentile_approx("account", 0.25).alias("bet_p25"),
    percentile_approx("account", 0.5).alias("bet_median"),
    percentile_approx("account", 0.75).alias("bet_p75"),

    Fmin("basepoint").alias("basepoint_min"),
    Fmax("basepoint").alias("basepoint_max"),
    Favg("basepoint").alias("basepoint_mean"),
    percentile_approx("basepoint", 0.25).alias("basepoint_p25"),
    percentile_approx("basepoint", 0.5).alias("basepoint_median"),
    percentile_approx("basepoint", 0.75).alias("basepoint_p75"),

    Fmin("payout").alias("payout_min"),
    Fmax("payout").alias("payout_max"),
    Favg("payout").alias("payout_mean"),
    percentile_approx("payout", 0.25).alias("payout_p25"),
    percentile_approx("payout", 0.5).alias("payout_median"),
    percentile_approx("payout", 0.75).alias("payout_p75"),

    Fmin("cus_account").alias("profit_min"),
    Fmax("cus_account").alias("profit_max"),
    Favg("cus_account").alias("profit_mean"),
    percentile_approx("cus_account", 0.25).alias("profit_p25"),
    percentile_approx("cus_account", 0.5).alias("profit_median"),
    percentile_approx("cus_account", 0.75).alias("profit_p75"),

    Fmin("delta_t").alias("delta_t_min"),
    Fmax("delta_t").alias("delta_t_max"),
    Favg("delta_t").alias("delta_t_mean"),
    percentile_approx("delta_t", 0.25).alias("delta_t_p25"),
    percentile_approx("delta_t", 0.5).alias("delta_t_median"),
    percentile_approx("delta_t", 0.75).alias("delta_t_p75"),

    Fmin("delta_bet").alias("delta_bet_min"),
    Fmax("delta_bet").alias("delta_bet_max"),
    Favg("delta_bet").alias("delta_bet_mean"),
    percentile_approx("delta_bet", 0.25).alias("delta_bet_p25"),
    percentile_approx("delta_bet", 0.5).alias("delta_bet_median"),
    percentile_approx("delta_bet", 0.75).alias("delta_bet_p75"),
    
    Fmin("delta_profit").alias("delta_profit_min"),
    Fmax("delta_profit").alias("delta_profit_max"),
    Favg("delta_profit").alias("delta_profit_mean"),
    percentile_approx("delta_profit", 0.25).alias("delta_profit_p25"),
    percentile_approx("delta_profit", 0.5).alias("delta_profit_median"),
    percentile_approx("delta_profit", 0.75).alias("delta_profit_p75"),
    
    Fmin("streak").alias("streak_min"),
    Fmax("streak").alias("streak_max"),
    Favg("streak").alias("streak_mean"),
    percentile_approx("streak", 0.25).alias("streak_p25"),
    percentile_approx("streak", 0.5).alias("streak_median"),
    percentile_approx("streak", 0.75).alias("streak_p75"),
    
    Fmin("win_streak").alias("win_streak_min"),
    Fmax("win_streak").alias("win_streak_max"),
    Favg("win_streak").alias("win_streak_mean"),
    percentile_approx("win_streak", 0.25).alias("win_streak_p25"),
    percentile_approx("win_streak", 0.5).alias("win_streak_median"),
    percentile_approx("win_streak", 0.75).alias("win_streak_p75"),
    
    
    Fmin("lose_streak").alias("lose_streak_min"),
    Fmax("lose_streak").alias("lose_streak_max"),
    Favg("lose_streak").alias("lose_streak_mean"),
    percentile_approx("lose_streak", 0.25).alias("lose_streak_p25"),
    percentile_approx("lose_streak", 0.5).alias("lose_streak_median"),
    percentile_approx("lose_streak", 0.75).alias("lose_streak_p75"),
    
    
    Fmin("deposit").alias("deposit_min"),
    Fmax("deposit").alias("deposit_max"),
    Favg("deposit").alias("deposit_mean"),
    percentile_approx("deposit", 0.25).alias("deposit_p25"),
    percentile_approx("deposit", 0.5).alias("deposit_median"),
    percentile_approx("deposit", 0.75).alias("deposit_p75"),
    
    Fmin("withdrawal").alias("withdrawal_min"),
    Fmax("withdrawal").alias("withdrawal_max"),
    Favg("withdrawal").alias("withdrawal_mean"),
    percentile_approx("withdrawal", 0.25).alias("withdrawal_p25"),
    percentile_approx("withdrawal", 0.5).alias("withdrawal_median"),
    percentile_approx("withdrawal", 0.75).alias("withdrawal_p75"),
    
     # slottype = 2.0 的数量
    Fsum(when(col("slottype") == 2.0, 1).otherwise(0)).alias("slottype_2_count"),

    # 获奖率：is_payout_gt0 的和 / group_num
    (Fsum("is_payout_gt0") / Fcount("*")).alias("payout_rate"),

    # 盈利率：is_profit_gt0 的和 / group_num
    (Fsum("is_profit_gt0") / Fcount("*")).alias("profit_rate"),

    # 盈利波动率：利润的标准差
    coalesce(stddev("cus_account"), lit(0)).alias("profit_stddev"),

    # 投注波动率：投注额的标准差
    coalesce(stddev("account"), lit(0)).alias("account_stddev"),

    # start_time（切片开始时间）
    Fmin("billtime").alias("start_time"),

    # end_time（切片结束时间）
    Fmax("billtime").alias("end_time"),
    #时间段
    Fsum("is_morning").alias("morning_count"),
    Fsum("is_afternoon").alias("afternoon_count"),
    Fsum("is_night").alias("night_count"),
    Fsum("is_midnight").alias("midnight_count"),
    Fsum("is_weekend").alias("weekend_count"),
    
    # 持续时间（秒）
    (unix_timestamp(Fmax("billtime")) - unix_timestamp(Fmin("billtime"))).alias("duration_seconds"),
    # 每注平均耗时（秒/注）
    ((unix_timestamp(Fmax("billtime")) - unix_timestamp(Fmin("billtime"))) / Fcount("*")).alias("avg_time_per_bet")
    
]
# 每个 group_id（loginname + group_index + sub_index）里统计 currency 出现次数
currency_count = df.groupBy("loginname", "group_index", "sub_index", "group_id", "currency_label", "currency").agg(
    Fcount("*").alias("currency_count")
)

# 取每个 group_id 出现最多的 currency
currency_window = Window.partitionBy("group_id").orderBy(col("currency_count").desc())

currency_count = currency_count.withColumn(
    "row_number",
    row_number().over(currency_window)
).filter(col("row_number") == 1)  # 只保留每组里出现次数最多的那一行

df_grouped = df.groupBy("loginname", "group_index", "sub_index", "group_id").agg(*agg_exprs)
# 把 currency_label 和 currency 也 join 回来
df_grouped = df_grouped.join(
    currency_count.select("group_id", "currency_label", "currency"),
    on="group_id",
    how="left"
)
# ========== 导出 ==========
df_grouped.write.mode("overwrite").option("header", True).csv("wucaishen_grouped_stat_output")
df.orderBy("loginname", "billtime").write.mode("overwrite").option("header", True).csv("wucaishen_enriched_output")
end_time = time.time()
print("Total execution time: {:.2f} seconds".format(end_time - start_time))

